In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
import os

In [97]:
# Cargamos los csv de los tifs
path = "saved_files/dataset"
dfs = {}
for archivo in os.listdir(path):
    if archivo.endswith("_features.csv"):
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        dfs[nombre_sin_extension[:-9]] = pd.read_csv(ruta_completa)

In [98]:
dfs_to_keep = [
    "C2X_3x3_merge_depth_lt_1", "C2X_5x5_merge_depth_lt_1", "C2X-Complex_5x5_merge_depth_lt_1", "C2X-Complex_3x3_merge_depth_lt_1",
    "C2X_3x3_merge_depth_gt_1", "C2X_5x5_merge_depth_gt_1", "C2X-Complex_5x5_merge_depth_gt_1", "C2X-Complex_3x3_merge_depth_gt_1",
    #"C2RCC_3x3_merge_depth_lt_1", "C2RCC_5x5_merge_depth_gt_1", "C2RCC_5x5_merge_depth_lt_1", "C2RCC_3x3_merge_depth_gt_1",
]

In [99]:
dfs = {k: dfs[k] for k in dfs_to_keep if k in dfs}

for nombre_df, df in dfs.items():
    for band_set in ["rhow", "rhown"]:
        dfs[nombre_df] = df.dropna()

In [68]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Invierno'
    elif month in [3, 4, 5]:
        return 'Primavera'
    elif month in [6, 7, 8]:
        return 'Verano'
    else:
        return 'Otoño'
    

def get_zone(buoy):
    if buoy in ["CTD1", "CTD2", "CTD3", "CTD4"]:
        return 'Zona-1'
    elif buoy in ["CTD6", "CTD8", "CTD9", "CTD10", "CTD12"]:
        return 'Zona-2'
    elif buoy in ["CTD7"]:
        return 'Zona-3'
    elif buoy in ["CTD11"]:
        return 'Zona-4'

In [100]:
for nombre_df, df in dfs.items():
    df["High_Chl"] = df["Chl"]>5
    df['Date'] = pd.to_datetime(df['Date'])
    df['Season'] = df['Date'].dt.month.apply(get_season)
    df['Zone'] = df['Buoy'].apply(get_zone)
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype('category')
    dfs[nombre_df] = df
    

In [103]:
def filtrar_columnas(df):

    # 1. Separar columnas numéricas y no numéricas
    df_numericas = df.select_dtypes(include='number')
    df_no_numericas = df.select_dtypes(exclude='number')

    # 2. Calcular la correlación con 'Chl' solo entre columnas numéricas
    correlaciones = df_numericas.corr()['Chl'].drop('Chl')

    # 3. Filtrar predictores numéricos con correlación significativa
    umbral_corr = 0.1
    columnas_utiles = correlaciones[correlaciones.abs() >= umbral_corr].index.tolist()

    # 4. Reconstruir el DataFrame con:
    # - Las columnas numéricas útiles
    # - La columna objetivo 'Chl'
    df_filtrado = pd.concat([df[columnas_utiles + ['Chl']]], axis=1)

    # Calcular la matriz de correlación entre predictores
    corr_matrix = df_filtrado.drop(columns='Chl').corr().abs()

    # Seleccionar columnas a eliminar (altamente correlacionadas entre sí)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    columnas_redundantes = [col for col in upper.columns if any(upper[col] > 0.98)]

    # Eliminar redundantes
    df_final = pd.concat([df_filtrado.drop(columns=columnas_redundantes), df_no_numericas], axis=1)
    df_final = df_final.drop(columns=["Date", "Buoy"])

    return df_final


In [104]:
for nombre_df, df in dfs.items():
    dfs[nombre_df] = filtrar_columnas(df)

In [105]:
df = dfs["C2X_3x3_merge_depth_lt_1"]

### Selección de hiperparámetros con Optuna

In [80]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score, KFold, train_test_split
import optuna
import time
from sklearn.model_selection import StratifiedKFold

In [128]:
train, test = train_test_split(df, test_size=0.25, random_state=42, stratify=df["High_Chl"])


In [129]:
train = train.reset_index()
test = test.reset_index()

In [113]:
def cross_validation(trial, df, target, model_name):
    
    # Separamos en X e y
    #X = df.loc[:, df.columns != target]
    X = df.drop(columns=[target, 'High_Chl'])
    y = df[target]
    y_class = df["High_Chl"]
    
    # Definimos los folds
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(df))  # Almacenar las predicciones OOF

    start = time.time()
    
    if model_name == "LBM":
        params_lbm = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'device': 'cpu',  # usa GPU
            'verbosity': -1,
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_categorical('num_leaves', [20, 40, 60, 80, 100]),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000]),
            'early_stopping_rounds': 50,
        }

    if model_name == "XGB":
        params_xgb = {
            'n_estimators': trial.suggest_categorical('n_estimators', [500, 1000, 2000]),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 5, 8),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 4),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'device': 'cpu',
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'enable_categorical': True,
            'early_stopping_rounds': 50,
            'eval_metric': 'rmse'
        }
    
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_class)):
        print(f"Fold {fold+1}")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if model_name == "XGB":
            model = XGBRegressor(**params_xgb)
            model.fit(X_train, y_train, 
                      eval_set=[(X_val, y_val)], 
                      verbose=0)
            
        # if model_name == "LBM":
        #     model = LGBMRegressor(**params_lbm)
        #     model.fit(X_train, y_train,
        #         eval_set=[(X_val, y_val)])
        
        # Guardamos las predicciones en su sitio correspondiente
        oof_preds[val_idx] = model.predict(X_val)

    print(f"Running time: {time.time() - start:.1f} sec")
    # Calculamos el RMSE OOF
    rmse_score = np.sqrt(mean_squared_error(y, oof_preds))
    print(f"OOF RMSE: {rmse_score:.4f}")
    
    return rmse_score

In [114]:
def run_optuna(df, target, n_trials, model_names):
    results = {}

    for model_name in model_names:
        print(f"Buscando mejores hiperparámetros para {model_name}...")
        study = optuna.create_study(direction='minimize')
        study.optimize(lambda trial: cross_validation(trial, df, target, model_name), n_trials=n_trials)
        print(f"\n✅ {model_name} - Mejor RMSLE: {study.best_value:.4f}")
        print(f"📋 Parámetros: {study.best_params}\n")
        
        results[model_name] = {
            'best_params': study.best_params,
            'best_score': study.best_value,
            'study': study
        }
    return results

In [115]:
model_names = ["XGB"]
n_trials = 20
results = run_optuna(train, "Chl", n_trials, model_names)

[I 2025-06-23 12:51:17,701] A new study created in memory with name: no-name-04ae869b-7568-4626-8d6b-7230bc9e55c8


Buscando mejores hiperparámetros para XGB...
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-06-23 12:51:21,177] Trial 0 finished with value: 2.698420717502573 and parameters: {'n_estimators': 1000, 'learning_rate': 0.007553279126968153, 'max_depth': 6, 'min_child_weight': 3, 'subsample': 0.6743716102734026, 'colsample_bytree': 0.7958130441878293}. Best is trial 0 with value: 2.698420717502573.


Running time: 3.5 sec
OOF RMSE: 2.6984
Fold 1
Fold 2
Fold 3
Fold 4


[I 2025-06-23 12:51:22,211] Trial 1 finished with value: 2.698087780292114 and parameters: {'n_estimators': 2000, 'learning_rate': 0.031978775110191315, 'max_depth': 5, 'min_child_weight': 3, 'subsample': 0.6496358282419495, 'colsample_bytree': 0.6337524652667789}. Best is trial 1 with value: 2.698087780292114.


...

Fold 5
Running time: 1.3 sec
OOF RMSE: 2.6515
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-06-23 12:51:57,944] Trial 18 finished with value: 2.583746560240734 and parameters: {'n_estimators': 2000, 'learning_rate': 0.04169919700025316, 'max_depth': 6, 'min_child_weight': 1, 'subsample': 0.9269056149463748, 'colsample_bytree': 0.6796424133878236}. Best is trial 12 with value: 2.5666571049859948.


Running time: 1.2 sec
OOF RMSE: 2.5837
Fold 1
Fold 2
Fold 3
Fold 4
Fold 5


[I 2025-06-23 12:51:59,413] Trial 19 finished with value: 2.5611457375564384 and parameters: {'n_estimators': 2000, 'learning_rate': 0.04814113153747816, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.9437728415677074, 'colsample_bytree': 0.909668290806239}. Best is trial 19 with value: 2.5611457375564384.


Running time: 1.5 sec
OOF RMSE: 2.5611

✅ XGB - Mejor RMSLE: 2.5611
📋 Parámetros: {'n_estimators': 2000, 'learning_rate': 0.04814113153747816, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.9437728415677074, 'colsample_bytree': 0.909668290806239}



In [111]:
best_params = {'n_estimators': 2000, 'learning_rate': 0.04814113153747816, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.9437728415677074, 'colsample_bytree': 0.909668290806239}
best_params = {'n_estimators': 2000, 'learning_rate': 0.048, 'max_depth': 7, 'min_child_weight': 1, 'subsample': 0.95, 'colsample_bytree': 0.9}

In [116]:
params_xgb= {
    'n_estimators': 2000,
    'learning_rate': 0.048,
    'max_depth': 7,
    'min_child_weight': 1,
    'subsample': 0.95,
    'colsample_bytree': 0.9,
    'device': 'cpu',
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'enable_categorical': True,
    'early_stopping_rounds': 50,
    'eval_metric': 'rmse'}

In [120]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_log_error
#from catboost import CatBoostRegressor
from xgboost import XGBRegressor
#from lightgbm import LGBMRegressor
import time
import joblib

### Entrenamiento XGB

In [133]:
save_folder = "training_results"
FOLDS = 5
kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

models = {
    #'LightGBM': LGBMRegressor(**params_lbm),
    'XGBoost': XGBRegressor(**params_xgb)
}
models_storage = {name: [] for name in models}

results = {name: {'oof': np.zeros(len(train)), 'pred': np.zeros(len(test)), 'rmse': [], 'r2': []} for name in models}

target = "Chl"
X = train.drop(columns=[target, 'High_Chl'])
X_test = test.drop(columns=[target, 'High_Chl'])
y = train[target]
y_class = train["High_Chl"]

    
# Definimos los folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\n=== Training {name} ===")
    for i, (train_idx, valid_idx) in enumerate(skf.split(X, y_class)):
        print(f"\nFold {i+1}")
        x_train, y_train = X.iloc[train_idx], y[train_idx]
        x_valid, y_valid = X.iloc[valid_idx], y[valid_idx]
        
        #x_train = x_train.loc[:, ~x_train.columns.duplicated()]
        #x_valid = x_valid.loc[:, ~x_valid.columns.duplicated()]
        #x_test = X_test.loc[:, ~X_test.columns.duplicated()].copy()

        start = time.time()
        
        if name == 'XGBoost':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=500)
        elif name == 'LightGBM':
            model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)])

        models_storage[name].append(model)
        #joblib.dump(model, f"{name}_fold_{i}.pkl") 
        oof_pred = model.predict(x_valid)
        test_pred = model.predict(X_test)
        
        results[name]['oof'][valid_idx] = oof_pred
        results[name]['pred'] += test_pred / FOLDS
        
        rmse = np.sqrt(mean_squared_error(y_valid, oof_pred))
        r2 = r2_score(y_valid, oof_pred)
        results[name]['rmse'].append(rmse)
        results[name]['r2'].append(r2)
        
        print(f"Fold {i+1} RMSE: {rmse:.2f} | R2: {r2:.2f}")
        print(f"Training time: {time.time() - start:.1f} sec")
    
    np.save(f"{save_folder}/{name}_oof.npy", results[name]['oof'])
    np.save(f"{save_folder}/{name}_pred.npy", results[name]['pred'])


print("\n=== Model Comparison ===")
for name in models:
    mean_rmse = np.mean(results[name]['rmse'])
    std_rmse = np.std(results[name]['rmse'])
    print(f"{name} - Mean RMSE: {mean_rmse:.2f} ± {std_rmse:.2f}")
    mean_r2 = np.mean(results[name]['r2'])
    std_r2 = np.std(results[name]['r2'])
    print(f"{name} - Mean R2: {mean_r2:.2f} ± {std_r2:.2f}")


=== Training XGBoost ===

Fold 1
[0]	validation_0-rmse:4.01941


[165]	validation_0-rmse:2.22910
Fold 1 RMSE: 2.23 | R2: 0.71
Training time: 0.4 sec

Fold 2
[0]	validation_0-rmse:2.96788
[483]	validation_0-rmse:1.76283
Fold 2 RMSE: 1.76 | R2: 0.66
Training time: 1.1 sec

Fold 3
[0]	validation_0-rmse:3.93877
[148]	validation_0-rmse:1.34212
Fold 3 RMSE: 1.34 | R2: 0.89
Training time: 0.4 sec

Fold 4
[0]	validation_0-rmse:3.53866
[334]	validation_0-rmse:1.15747
Fold 4 RMSE: 1.16 | R2: 0.90
Training time: 0.9 sec

Fold 5
[0]	validation_0-rmse:3.08283
[452]	validation_0-rmse:1.71500
Fold 5 RMSE: 1.71 | R2: 0.71
Training time: 1.1 sec

=== Model Comparison ===
XGBoost - Mean RMSE: 1.64 ± 0.37
XGBoost - Mean R2: 0.78 ± 0.10
